# Protección de Datos Sensibles (PII) en Sistemas LLM

> Aprende a detectar, anonimizar y gestionar datos personales (PII) antes de enviarlos
> a modelos de IA externos. Cumplimiento GDPR en la práctica.

## Objetivos
- Detectar PII con Presidio: emails, teléfonos, DNIs, nombres, tarjetas
- Anonimizar texto antes de enviarlo a la API
- Construir un pipeline que anonimiza → procesa → de-anonimiza
- Implementar un logger seguro que redacta PII automáticamente

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic presidio-analyzer presidio-anonymizer spacy --quiet
# Descargar modelo de idioma para Presidio
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "es_core_news_sm"], check=False)
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=False)

## 2. Configurar Presidio para detección de PII

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import anthropic

# Inicializar motores de Presidio
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()
client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Texto con múltiples tipos de PII
TEXTO_CON_PII = """
Buenos días, soy María González (mgonzalez@empresa.com).
Mi DNI es 12345678A y mi teléfono es +34 612 345 678.
El número de mi tarjeta es 4532-1234-5678-9012 (vence 12/26, CVV 123).
Necesito ayuda con mi cuenta bancaria ES91 2100 0418 4502 0005 1332.
Mi dirección: Calle Mayor 15, 3ºB, 28013 Madrid.
"""

print("Texto original con PII:")
print(TEXTO_CON_PII)

## 3. Detección y categorización de PII

In [ ]:
# Detectar PII en español e inglés
resultados_analisis = analyzer.analyze(
    text=TEXTO_CON_PII,
    language="es",
    entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD",
               "IBAN_CODE", "LOCATION", "NRP"]
)

print(f"=== PII DETECTADA ({len(resultados_analisis)} entidades) ===")
for resultado in sorted(resultados_analisis, key=lambda x: x.start):
    fragmento = TEXTO_CON_PII[resultado.start:resultado.end]
    print(f"  Tipo: {resultado.entity_type:20s} | Confianza: {resultado.score:.2f} | Valor: '{fragmento}'")

## 4. Anonimización automática

In [ ]:
# Anonimizar reemplazando cada tipo con un token descriptivo
operadores = {
    "PERSON": OperatorConfig("replace", {"new_value": "[NOMBRE]"}),
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"}),
    "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "[TELÉFONO]"}),
    "CREDIT_CARD": OperatorConfig("replace", {"new_value": "[TARJETA]"}),
    "IBAN_CODE": OperatorConfig("replace", {"new_value": "[IBAN]"}),
    "LOCATION": OperatorConfig("replace", {"new_value": "[DIRECCIÓN]"}),
    "NRP": OperatorConfig("replace", {"new_value": "[DNI]"})
}

texto_anonimizado = anonymizer.anonymize(
    text=TEXTO_CON_PII,
    analyzer_results=resultados_analisis,
    operators=operadores
)

print("=== TEXTO ANONIMIZADO (seguro para enviar a API externa) ===")
print(texto_anonimizado.text)

## 5. Pipeline: anonimizar → LLM → de-anonimizar

In [ ]:
def procesar_con_privacidad(texto: str, tarea: str) -> str:
    """Procesa texto con LLM externo sin exponer PII."""
    # 1. Detectar PII
    entidades = analyzer.analyze(text=texto, language="es")

    # 2. Crear mapa de sustituciones para revertir después
    mapa_sustitucion = {}
    for i, entidad in enumerate(entidades):
        token = f"[PII_{entidad.entity_type}_{i}]"
        mapa_sustitucion[token] = texto[entidad.start:entidad.end]

    # 3. Anonimizar
    ops = {e.entity_type: OperatorConfig("replace", {"new_value": f"[PII_{e.entity_type}_{i}]"})
           for i, e in enumerate(entidades)}
    resultado_anon = anonymizer.anonymize(text=texto, analyzer_results=entidades, operators=ops)
    texto_anon = resultado_anon.text

    # 4. Enviar texto anonimizado al LLM
    respuesta = client.messages.create(
        model=MODELO, max_tokens=512,
        messages=[{"role": "user", "content": f"{tarea}:\n{texto_anon}"}]
    ).content[0].text

    # 5. De-anonimizar la respuesta (restaurar valores originales)
    for token, valor_original in mapa_sustitucion.items():
        respuesta = respuesta.replace(token, valor_original)

    return respuesta

EMAIL_CLIENTE = """
De: carlos.ruiz@gmail.com
Asunto: Problema con pedido

Hola, soy Carlos Ruiz (Tel: +34 699 123 456). Hice un pedido el pasado lunes
y aún no ha llegado. Mi número de pedido es #87234. ¿Pueden ayudarme?
"""

print("=== PIPELINE PRIVACIDAD-PRIMERO ===")
print("Input original (con PII):")
print(EMAIL_CLIENTE)

respuesta = procesar_con_privacidad(
    EMAIL_CLIENTE,
    "Resume este email de cliente y sugiere una respuesta profesional"
)
print("\nRespuesta del LLM (PII restaurada):")
print(respuesta)